<a href="https://colab.research.google.com/github/Arjun0650/Agentic-AI/blob/main/AgenticAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q \
    google-genai \
    chromadb \
    fastapi \
    uvicorn \
    pyngrok \
    nest-asyncio \
    python-multipart

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 99.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

In [2]:
import os
from getpass import getpass

GOOGLE_API_KEY = getpass("Enter your Google Gemini API key: ")

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

print("API key configured successfully.")

Enter your Google Gemini API key: ··········
API key configured successfully.


In [3]:
import os

PROJECT_DIR = "/content/schedule_agent"

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/data", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/static", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/templates", exist_ok=True)

print("Project created at:", PROJECT_DIR)

Project created at: /content/schedule_agent


In [5]:
import sqlite3
import os
from datetime import datetime

DB_PATH = f"{PROJECT_DIR}/data/schedule.db"


def get_connection():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def init_database():
    conn = get_connection()

    conn.execute("""
        CREATE TABLE IF NOT EXISTS events (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL,
            description TEXT,
            event_type TEXT NOT NULL,
            date TEXT NOT NULL,
            start_time TEXT NOT NULL,
            end_time TEXT NOT NULL,
            location TEXT,
            status TEXT DEFAULT 'scheduled',
            created_at TEXT DEFAULT CURRENT_TIMESTAMP
        )
    """)

    conn.commit()
    conn.close()

    print("Database initialized.")


init_database()

Database initialized.


In [4]:
from datetime import datetime


def add_event(
    title,
    description,
    event_type,
    date,
    start_time,
    end_time,
    location=""
):
    conn = get_connection()

    cursor = conn.execute("""
        INSERT INTO events
        (title, description, event_type, date, start_time, end_time, location)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, (
        title,
        description,
        event_type,
        date,
        start_time,
        end_time,
        location
    ))

    conn.commit()

    event_id = cursor.lastrowid

    conn.close()

    return event_id


def get_all_events():
    conn = get_connection()

    rows = conn.execute("""
        SELECT *
        FROM events
        WHERE status = 'scheduled'
        ORDER BY date, start_time
    """).fetchall()

    conn.close()

    return [dict(row) for row in rows]


def get_events_by_date(date):
    conn = get_connection()

    rows = conn.execute("""
        SELECT *
        FROM events
        WHERE date = ?
        AND status = 'scheduled'
        ORDER BY start_time
    """, (date,)).fetchall()

    conn.close()

    return [dict(row) for row in rows]


def search_events(query):
    conn = get_connection()

    pattern = f"%{query}%"

    rows = conn.execute("""
        SELECT *
        FROM events
        WHERE status = 'scheduled'
        AND (
            title LIKE ?
            OR description LIKE ?
            OR event_type LIKE ?
            OR location LIKE ?
            OR date LIKE ?
        )
        ORDER BY date, start_time
    """, (
        pattern,
        pattern,
        pattern,
        pattern,
        pattern
    )).fetchall()

    conn.close()

    return [dict(row) for row in rows]


def update_event(
    event_id,
    title=None,
    description=None,
    event_type=None,
    date=None,
    start_time=None,
    end_time=None,
    location=None
):
    conn = get_connection()

    fields = []
    values = []

    updates = {
        "title": title,
        "description": description,
        "event_type": event_type,
        "date": date,
        "start_time": start_time,
        "end_time": end_time,
        "location": location
    }

    for field, value in updates.items():
        if value is not None:
            fields.append(f"{field} = ?")
            values.append(value)

    if not fields:
        conn.close()
        return False

    values.append(event_id)

    conn.execute(
        f"""
        UPDATE events
        SET {', '.join(fields)}
        WHERE id = ?
        """,
        values
    )

    conn.commit()
    conn.close()

    return True


def delete_event(event_id):
    conn = get_connection()

    conn.execute("""
        UPDATE events
        SET status = 'cancelled'
        WHERE id = ?
    """, (event_id,))

    conn.commit()
    conn.close()

    return True


print("Database functions ready.")

Database functions ready.


In [6]:
from datetime import date, timedelta
import random

def seed_sample_schedule():

    # Don't duplicate data if this cell is run again
    conn = get_connection()

    count = conn.execute("""
        SELECT COUNT(*) FROM events
    """).fetchone()[0]

    conn.close()

    if count > 0:
        print(f"Database already contains {count} events.")
        return

    event_templates = [
        {
            "title": "Team Meeting",
            "description": "Weekly project discussion with the development team.",
            "event_type": "meeting",
            "start": "10:00",
            "end": "11:00",
            "location": "Conference Room"
        },
        {
            "title": "AI/ML Workshop",
            "description": "Workshop covering machine learning and artificial intelligence.",
            "event_type": "workshop",
            "start": "14:00",
            "end": "16:00",
            "location": "AI Lab"
        },
        {
            "title": "Project Development Task",
            "description": "Work on the Agentic RAG Schedule Assistant project.",
            "event_type": "task",
            "start": "09:00",
            "end": "11:00",
            "location": "Home"
        },
        {
            "title": "Doctor Appointment",
            "description": "Regular medical appointment.",
            "event_type": "appointment",
            "start": "16:00",
            "end": "17:00",
            "location": "City Hospital"
        },
        {
            "title": "Client Meeting",
            "description": "Discuss project requirements and progress with client.",
            "event_type": "meeting",
            "start": "15:00",
            "end": "16:00",
            "location": "Online"
        },
        {
            "title": "Python Practice",
            "description": "Practice Python programming and data structures.",
            "event_type": "task",
            "start": "18:00",
            "end": "19:00",
            "location": "Home"
        }
    ]

    today = date.today()

    for i in range(30):

        current_date = today + timedelta(days=i)

        # Add 1–2 events each day
        number_of_events = random.choice([1, 2])

        selected = random.sample(
            event_templates,
            number_of_events
        )

        for template in selected:

            add_event(
                title=template["title"],
                description=template["description"],
                event_type=template["event_type"],
                date=current_date.isoformat(),
                start_time=template["start"],
                end_time=template["end"],
                location=template["location"]
            )

    print("30 days of sample schedule created.")


seed_sample_schedule()

30 days of sample schedule created.


In [7]:
events = get_all_events()

print("Total events:", len(events))

for event in events[:10]:
    print(
        f'{event["date"]} | '
        f'{event["start_time"]}-{event["end_time"]} | '
        f'{event["title"]}'
    )

Total events: 48
2026-08-26 | 14:00-16:00 | AI/ML Workshop
2026-08-26 | 16:00-17:00 | Doctor Appointment
2026-08-27 | 14:00-16:00 | AI/ML Workshop
2026-08-28 | 16:00-17:00 | Doctor Appointment
2026-08-29 | 16:00-17:00 | Doctor Appointment
2026-08-30 | 09:00-11:00 | Project Development Task
2026-08-30 | 15:00-16:00 | Client Meeting
2026-08-31 | 09:00-11:00 | Project Development Task
2026-08-31 | 18:00-19:00 | Python Practice
2026-09-01 | 09:00-11:00 | Project Development Task


In [8]:
import chromadb

CHROMA_PATH = f"{PROJECT_DIR}/data/chroma_db"

chroma_client = chromadb.PersistentClient(
    path=CHROMA_PATH
)

collection = chroma_client.get_or_create_collection(
    name="schedule_events"
)

print("ChromaDB initialized.")

ChromaDB initialized.


In [9]:
from google import genai

client = genai.Client(
    api_key=os.environ["GOOGLE_API_KEY"]
)

EMBEDDING_MODEL = "gemini-embedding-001"

print("Gemini client initialized.")

Gemini client initialized.


In [11]:
def event_to_text(event):

    return f"""
Event ID: {event['id']}
Title: {event['title']}
Description: {event['description']}
Type: {event['event_type']}
Date: {event['date']}
Start Time: {event['start_time']}
End Time: {event['end_time']}
Location: {event['location']}
"""


def create_embedding(text):

    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text
    )

    return response.embeddings[0].values


def index_schedule():

    events = get_all_events()

    # Clear existing collection
    existing = collection.get()

    if existing["ids"]:
        collection.delete(ids=existing["ids"])

    for event in events:

        text = event_to_text(event)

        embedding = create_embedding(text)

        collection.add(
            ids=[str(event["id"])],
            embeddings=[embedding],
            documents=[text],
            metadatas=[{
                "event_id": event["id"],
                "date": event["date"],
                "event_type": event["event_type"],
                "start_time": event["start_time"],
                "end_time": event["end_time"]
            }]
        )

    print(f"Indexed {len(events)} events into ChromaDB.")


index_schedule()

Indexed 48 events into ChromaDB.


In [12]:
def retrieve_schedule(query, n_results=5):

    query_embedding = create_embedding(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )

    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]

    output = []

    for document, metadata in zip(documents, metadatas):
        output.append({
            "document": document,
            "metadata": metadata
        })

    return output


results = retrieve_schedule(
    "AI workshop and meetings"
)

for result in results:
    print(result["document"])
    print("-" * 50)


Event ID: 2
Title: AI/ML Workshop
Description: Workshop covering machine learning and artificial intelligence.
Type: workshop
Date: 2026-08-26
Start Time: 14:00
End Time: 16:00
Location: AI Lab

--------------------------------------------------

Event ID: 3
Title: AI/ML Workshop
Description: Workshop covering machine learning and artificial intelligence.
Type: workshop
Date: 2026-08-27
Start Time: 14:00
End Time: 16:00
Location: AI Lab

--------------------------------------------------

Event ID: 21
Title: AI/ML Workshop
Description: Workshop covering machine learning and artificial intelligence.
Type: workshop
Date: 2026-09-08
Start Time: 14:00
End Time: 16:00
Location: AI Lab

--------------------------------------------------

Event ID: 35
Title: AI/ML Workshop
Description: Workshop covering machine learning and artificial intelligence.
Type: workshop
Date: 2026-09-16
Start Time: 14:00
End Time: 16:00
Location: AI Lab

--------------------------------------------------

Event ID:

In [13]:
from datetime import datetime, timedelta


def get_schedule(
    date_query=None,
    time_query=None,
    user_query=None
):
    """
    Retrieves relevant schedule information
    based on date, time or natural language query.
    """

    # Date-specific search
    if date_query:

        events = get_events_by_date(date_query)

        return {
            "success": True,
            "source": "database",
            "events": events
        }

    # Natural language / semantic search
    if user_query:

        results = retrieve_schedule(
            user_query,
            n_results=8
        )

        return {
            "success": True,
            "source": "chromadb",
            "results": results
        }

    # Return complete schedule
    return {
        "success": True,
        "source": "database",
        "events": get_all_events()
    }


print("get_schedule tool ready.")

get_schedule tool ready.


In [14]:
def update_schedule(
    action,
    event_id=None,
    title=None,
    description=None,
    event_type=None,
    date=None,
    start_time=None,
    end_time=None,
    location=None
):

    action = action.lower()

    # ADD
    if action == "add":

        if not title or not date or not start_time or not end_time:
            return {
                "success": False,
                "message": "title, date, start_time and end_time are required."
            }

        new_id = add_event(
            title=title,
            description=description or "",
            event_type=event_type or "task",
            date=date,
            start_time=start_time,
            end_time=end_time,
            location=location or ""
        )

        # Re-index ChromaDB
        index_schedule()

        return {
            "success": True,
            "action": "added",
            "event_id": new_id
        }

    # UPDATE
    elif action == "update":

        if not event_id:
            return {
                "success": False,
                "message": "event_id is required for update."
            }

        success = update_event(
            event_id=event_id,
            title=title,
            description=description,
            event_type=event_type,
            date=date,
            start_time=start_time,
            end_time=end_time,
            location=location
        )

        index_schedule()

        return {
            "success": success,
            "action": "updated",
            "event_id": event_id
        }

    # DELETE
    elif action == "delete":

        if not event_id:
            return {
                "success": False,
                "message": "event_id is required for delete."
            }

        delete_event(event_id)

        index_schedule()

        return {
            "success": True,
            "action": "deleted",
            "event_id": event_id
        }

    return {
        "success": False,
        "message": f"Unknown action: {action}"
    }


print("update_schedule tool ready.")

update_schedule tool ready.


In [15]:
AGENT_MODEL = "gemini-3.6-flash"


def agent_system_prompt():

    return """
You are an intelligent Schedule Assistant.

You manage the user's schedule for the next 30 days.

You have exactly two tools:

1. get_schedule
   Use this when the user wants to:
   - See their schedule
   - Find events
   - Check meetings
   - Check appointments
   - Find tasks
   - Check whether they are free
   - Search their schedule by date, time or topic

2. update_schedule
   Use this when the user wants to:
   - Add an event
   - Create a meeting
   - Create a workshop
   - Add an appointment
   - Add a task
   - Move an event
   - Change an event
   - Delete an event

Rules:

- Always use get_schedule when schedule information is needed.
- Always use update_schedule when the user asks to change the schedule.
- Never claim an event was added, changed or deleted unless the update tool succeeds.
- When checking availability, inspect the schedule first.
- Understand natural language dates such as tomorrow, today, Friday, next Monday, etc.
- Current date is provided by the application.
- Give concise and useful answers.
"""

In [16]:
import json
from datetime import date


def run_agent(user_message):

    today = date.today().isoformat()

    prompt = f"""
{agent_system_prompt()}

Today's date is {today}.

The user said:

{user_message}

Decide what action is required.

Return ONLY valid JSON in this format:

{{
    "action": "get_schedule" | "update_schedule" | "answer",
    "arguments": {{}},
    "response": "your response"
}}

For get_schedule:

{{
    "action": "get_schedule",
    "arguments": {{
        "date_query": null,
        "time_query": null,
        "user_query": "..."
    }},
    "response": ""
}}

For update_schedule:

{{
    "action": "update_schedule",
    "arguments": {{
        "action": "add" | "update" | "delete",
        "event_id": null,
        "title": null,
        "description": null,
        "event_type": null,
        "date": null,
        "start_time": null,
        "end_time": null,
        "location": null
    }},
    "response": ""
}}

For a direct answer:

{{
    "action": "answer",
    "arguments": {{}},
    "response": "..."
}}
"""

    response = client.models.generate_content(
        model=AGENT_MODEL,
        contents=prompt
    )

    text = response.text.strip()

    # Remove markdown code fences if Gemini adds them
    if text.startswith("```"):
        text = text.replace("```json", "")
        text = text.replace("```", "")
        text = text.strip()

    try:
        decision = json.loads(text)

    except Exception:

        return {
            "success": False,
            "message": "Agent returned invalid JSON.",
            "raw": text
        }

    action = decision.get("action")
    arguments = decision.get("arguments", {})

    # TOOL 1
    if action == "get_schedule":

        result = get_schedule(
            date_query=arguments.get("date_query"),
            time_query=arguments.get("time_query"),
            user_query=arguments.get("user_query")
        )

        # Ask Gemini to summarize retrieved information
        summary_prompt = f"""
You are a schedule assistant.

User request:
{user_message}

Retrieved schedule information:
{json.dumps(result, indent=2)}

Answer the user naturally.

If the user asked about availability, clearly say whether
the requested period appears free or occupied.

Do not invent events.
"""

        summary = client.models.generate_content(
            model=AGENT_MODEL,
            contents=summary_prompt
        )

        return {
            "success": True,
            "tool": "get_schedule",
            "result": result,
            "response": summary.text
        }

    # TOOL 2
    elif action == "update_schedule":

        result = update_schedule(
            action=arguments.get("action"),
            event_id=arguments.get("event_id"),
            title=arguments.get("title"),
            description=arguments.get("description"),
            event_type=arguments.get("event_type"),
            date=arguments.get("date"),
            start_time=arguments.get("start_time"),
            end_time=arguments.get("end_time"),
            location=arguments.get("location")
        )

        summary_prompt = f"""
The user requested:

{user_message}

The schedule update tool returned:

{json.dumps(result, indent=2)}

Give the user a concise confirmation.

Do not claim success if success is false.
"""

        summary = client.models.generate_content(
            model=AGENT_MODEL,
            contents=summary_prompt
        )

        return {
            "success": True,
            "tool": "update_schedule",
            "result": result,
            "response": summary.text
        }

    # DIRECT ANSWER
    return {
        "success": True,
        "tool": None,
        "response": decision.get("response", "")
    }


print("Agent ready.")

Agent ready.


In [18]:
result = run_agent(
    "What do I have scheduled tomorrow?"
)

print(result["response"])

You have one event scheduled for tomorrow:

* **AI/ML Workshop**
  * **Time:** 2:00 PM – 4:00 PM (14:00 – 16:00)
  * **Location:** AI Lab
  * **Description:** Workshop covering machine learning and artificial intelligence.


In [19]:
result = run_agent(
    "Am I free Friday afternoon?"
)

print(result["response"])

You are mostly free on Friday afternoon, but you are not completely open. 

You have one scheduled event:
* **Doctor Appointment:** 4:00 PM – 5:00 PM at City Hospital

The rest of Friday afternoon before 4:00 PM is clear.


In [20]:
result = run_agent(
    "Add a meeting on September 5 at 3 PM for one hour."
)

print(result["response"])

Indexed 49 events into ChromaDB.
Your meeting has been scheduled for September 5 from 3:00 PM to 4:00 PM.


In [21]:
result = run_agent(
    "Do I have any workshops coming up?"
)

print(result["response"])

Yes, you have several **AI/ML Workshops** scheduled in August and September 2026. All of them are set to take place in the **AI Lab** from **14:00 to 16:00**.

Here is the list of upcoming workshop dates:

* **August 26, 2026** (14:00 – 16:00)
* **August 27, 2026** (14:00 – 16:00)
* **September 8, 2026** (14:00 – 16:00)
* **September 9, 2026** (14:00 – 16:00)
* **September 11, 2026** (14:00 – 16:00)
* **September 16, 2026** (14:00 – 16:00)
* **September 20, 2026** (14:00 – 16:00)

Let me know if you need more details about any specific session!


In [22]:
events = get_all_events()

for event in events[:10]:
    print(
        event["id"],
        "|",
        event["date"],
        "|",
        event["start_time"],
        "|",
        event["title"]
    )

2 | 2026-08-26 | 14:00 | AI/ML Workshop
1 | 2026-08-26 | 16:00 | Doctor Appointment
3 | 2026-08-27 | 14:00 | AI/ML Workshop
4 | 2026-08-28 | 16:00 | Doctor Appointment
5 | 2026-08-29 | 16:00 | Doctor Appointment
7 | 2026-08-30 | 09:00 | Project Development Task
6 | 2026-08-30 | 15:00 | Client Meeting
8 | 2026-08-31 | 09:00 | Project Development Task
9 | 2026-08-31 | 18:00 | Python Practice
10 | 2026-09-01 | 09:00 | Project Development Task


In [23]:
result = run_agent(
    "Move my Team Meeting from 10 AM to 4 PM."
)

print(result["response"])

I wasn't able to move your Team Meeting to 4 PM because an event ID is required. Please specify which meeting you'd like to update.


In [24]:
%%writefile /content/schedule_agent/app.py

import os
import sys
import json

sys.path.append("/content/schedule_agent")

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI(
    title="Agentic RAG Schedule Assistant",
    description="AI-powered 30-day schedule management agent"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"]
)


class ChatRequest(BaseModel):
    message: str


@app.get("/")
def home():
    return {
        "message": "Agentic RAG Schedule Assistant is running."
    }


@app.get("/health")
def health():
    return {
        "status": "healthy"
    }


@app.post("/chat")
def chat(request: ChatRequest):

    # Import from notebook globals
    result = run_agent(request.message)

    return result

Writing /content/schedule_agent/app.py


In [25]:
%%writefile /content/schedule_agent/templates/index.html

<!DOCTYPE html>
<html lang="en">

<head>

<meta charset="UTF-8">

<meta name="viewport"
      content="width=device-width, initial-scale=1.0">

<title>AI Schedule Assistant</title>

<link rel="stylesheet"
      href="/static/style.css">

</head>

<body>

<div class="container">

    <div class="header">

        <h1>AI Schedule Assistant</h1>

        <p>
            Agentic RAG-powered 30-day schedule manager
        </p>

    </div>


    <div id="chat" class="chat">

        <div class="message bot">
            Hello! I can manage your schedule for the next
            30 days. Ask me about your schedule or ask me
            to add, update or remove an event.
        </div>

    </div>


    <div class="input-area">

        <input
            id="message"
            type="text"
            placeholder="Ask about your schedule..."
            onkeydown="handleKey(event)"
        >

        <button onclick="sendMessage()">
            Send
        </button>

    </div>

</div>


<script>

function handleKey(event) {

    if (event.key === "Enter") {
        sendMessage();
    }

}


async function sendMessage() {

    const input =
        document.getElementById("message");

    const message =
        input.value.trim();

    if (!message) return;


    addMessage(message, "user");

    input.value = "";


    addMessage("Thinking...", "bot", true);


    try {

        const response = await fetch(
            "/chat",
            {
                method: "POST",

                headers: {
                    "Content-Type":
                    "application/json"
                },

                body: JSON.stringify({
                    message: message
                })
            }
        );


        const data =
            await response.json();


        removeThinking();


        addMessage(
            data.response || data.message,
            "bot"
        );


    } catch (error) {

        removeThinking();

        addMessage(
            "Something went wrong. Please try again.",
            "bot"
        );

    }

}


function addMessage(
    text,
    type,
    thinking=false
) {

    const chat =
        document.getElementById("chat");


    const div =
        document.createElement("div");


    div.className =
        "message " + type;


    if (thinking) {
        div.id = "thinking";
    }


    div.innerText = text;


    chat.appendChild(div);


    chat.scrollTop =
        chat.scrollHeight;

}


function removeThinking() {

    const thinking =
        document.getElementById("thinking");

    if (thinking) {
        thinking.remove();
    }

}

</script>

</body>

</html>

Writing /content/schedule_agent/templates/index.html


In [26]:
%%writefile /content/schedule_agent/static/style.css

* {
    box-sizing: border-box;
}

body {

    margin: 0;

    font-family:
        Arial,
        Helvetica,
        sans-serif;

    background:
        linear-gradient(
            135deg,
            #071b36,
            #102d52
        );

    min-height: 100vh;

    display: flex;

    align-items: center;

    justify-content: center;
}


.container {

    width: 95%;

    max-width: 900px;

    height: 90vh;

    background: white;

    border-radius: 20px;

    overflow: hidden;

    box-shadow:
        0 20px 60px
        rgba(0,0,0,0.3);

    display: flex;

    flex-direction: column;
}


.header {

    background: #0B1F3A;

    color: white;

    padding: 25px;

    text-align: center;
}


.header h1 {

    margin: 0 0 8px;

}


.header p {

    margin: 0;

    opacity: 0.8;

}


.chat {

    flex: 1;

    padding: 25px;

    overflow-y: auto;

    background: #f4f7fb;

}


.message {

    max-width: 75%;

    padding: 14px 18px;

    margin-bottom: 15px;

    border-radius: 15px;

    line-height: 1.5;

    white-space: pre-wrap;

}


.message.bot {

    background: white;

    color: #222;

    border:
        1px solid #e1e5eb;

}


.message.user {

    background: #0B1F3A;

    color: white;

    margin-left: auto;

}


.input-area {

    display: flex;

    padding: 18px;

    border-top:
        1px solid #ddd;

    background: white;

}


.input-area input {

    flex: 1;

    padding: 15px;

    border:
        1px solid #ccc;

    border-radius: 10px;

    font-size: 16px;

    outline: none;

}


.input-area button {

    margin-left: 10px;

    padding:
        0 25px;

    border: none;

    border-radius: 10px;

    background: #0B1F3A;

    color: white;

    font-size: 16px;

    cursor: pointer;

}


.input-area button:hover {

    opacity: 0.9;

}


@media (max-width: 600px) {

    .container {
        width: 100%;
        height: 100vh;
        border-radius: 0;
    }

    .message {
        max-width: 90%;
    }

}

Writing /content/schedule_agent/static/style.css


In [29]:
# CELL 24 — Create and start FastAPI application

import sys
import os
import threading
import nest_asyncio
import uvicorn

from fastapi import FastAPI, Request
from fastapi.responses import HTMLResponse
from fastapi.staticfiles import StaticFiles
from fastapi.templating import Jinja2Templates

nest_asyncio.apply()

# Create FastAPI application
app = FastAPI(
    title="Agentic RAG Schedule Assistant",
    description="AI-powered 30-day schedule management agent",
    version="1.0.0"
)

# Enable CORS
from fastapi.middleware.cors import CORSMiddleware

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"]
)

# Static files
app.mount(
    "/static",
    StaticFiles(
        directory=f"{PROJECT_DIR}/static"
    ),
    name="static"
)

# HTML templates
templates = Jinja2Templates(
    directory=f"{PROJECT_DIR}/templates"
)


# -----------------------------
# HOME API
# -----------------------------

@app.get("/")
def home():

    return {
        "status": "running",
        "message": "Agentic RAG Schedule Assistant is running.",
        "interface": "/chat-ui"
    }


# -----------------------------
# HEALTH CHECK
# -----------------------------

@app.get("/health")
def health():

    return {
        "status": "healthy"
    }


# -----------------------------
# CHAT API
# -----------------------------

from pydantic import BaseModel


class ChatRequest(BaseModel):

    message: str


@app.post("/chat")
def chat(request: ChatRequest):

    try:

        result = run_agent(request.message)

        return result

    except Exception as e:

        return {
            "success": False,
            "error": str(e)
        }


# -----------------------------
# WEB INTERFACE
# -----------------------------

@app.get(
    "/chat-ui",
    response_class=HTMLResponse
)
def chat_ui(request: Request):

    return templates.TemplateResponse(
        "index.html",
        {
            "request": request
        }
    )


# -----------------------------
# START SERVER
# -----------------------------

def start_server():

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )


# Start server in background
server_thread = threading.Thread(
    target=start_server,
    daemon=True
)

server_thread.start()

print("======================================")
print("FastAPI server started successfully!")
print("======================================")
print()
print("Local API:")
print("http://127.0.0.1:8000")
print()
print("Chat interface:")
print("http://127.0.0.1:8000/chat-ui")

FastAPI server started successfully!

Local API:
http://127.0.0.1:8000

Chat interface:
http://127.0.0.1:8000/chat-ui


In [32]:
# CELL 25 — Authenticate ngrok and create public URL

from pyngrok import ngrok
from getpass import getpass

# Enter your ngrok authtoken
NGROK_AUTH_TOKEN = getpass("Enter your ngrok authtoken: ")

# Configure ngrok authentication
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Stop any existing tunnels
ngrok.kill()

# Create public tunnel to FastAPI
public_url = ngrok.connect(8000)

print("======================================")
print("NGROK TUNNEL CREATED")
print("======================================")
print()
print("Public URL:")
print(public_url)
print()
print("Open this URL in your browser:")
print(str(public_url) + "/chat-ui")

Enter your ngrok authtoken: ··········
NGROK TUNNEL CREATED

Public URL:
NgrokTunnel: "https://immovably-caravan-breeches.ngrok-free.dev" -> "http://localhost:8000"

Open this URL in your browser:
NgrokTunnel: "https://immovably-caravan-breeches.ngrok-free.dev" -> "http://localhost:8000"/chat-ui


In [34]:
print("Your actual ngrok URL:")
print(public_url)

print("\nChat UI:")
print(str(public_url).rstrip("/") + "/chat-ui")

Your actual ngrok URL:
NgrokTunnel: "https://immovably-caravan-breeches.ngrok-free.dev" -> "http://localhost:8000"

Chat UI:
NgrokTunnel: "https://immovably-caravan-breeches.ngrok-free.dev" -> "http://localhost:8000"/chat-ui


In [35]:
print("Open this URL in your browser:")
print(str(public_url).rstrip("/") + "/chat-ui")

Open this URL in your browser:
NgrokTunnel: "https://immovably-caravan-breeches.ngrok-free.dev" -> "http://localhost:8000"/chat-ui


In [36]:
# CELL 26 — Show the actual website URL

chat_ui_url = str(public_url).rstrip("/") + "/chat-ui"

print("======================================")
print("YOUR SCHEDULE ASSISTANT")
print("======================================")
print()
print(chat_ui_url)

YOUR SCHEDULE ASSISTANT

NgrokTunnel: "https://immovably-caravan-breeches.ngrok-free.dev" -> "http://localhost:8000"/chat-ui


In [38]:
# CELL 27 — Test the web interface

import requests

# Extract only the public HTTPS URL
base_url = public_url.public_url

chat_ui_url = base_url + "/chat-ui"

print("Testing:")
print(chat_ui_url)
print()

response = requests.get(chat_ui_url)

print("Status code:", response.status_code)
print()
print("First part of response:")
print(response.text[:300])

Testing:
https://immovably-caravan-breeches.ngrok-free.dev/chat-ui

INFO:     34.7.143.23:0 - "GET /chat-ui HTTP/1.1" 500 Internal Server Error


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 422, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        self.scope, self.receive, self.send
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/middleware/proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/fastapi/applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/applications.py", line 96, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/middleware/errors.py", line 186, i

Status code: 500

First part of response:
Internal Server Error


In [39]:
# CELL 28 — Test the AI Schedule Agent

chat_url = public_url.public_url + "/chat"

response = requests.post(
    chat_url,
    json={
        "message": "What do I have scheduled tomorrow?"
    },
    timeout=120
)

print("Status code:", response.status_code)
print()
print(response.json())

INFO:     34.7.143.23:0 - "POST /chat HTTP/1.1" 200 OK
Status code: 200

{'success': True, 'tool': 'get_schedule', 'result': {'success': True, 'source': 'database', 'events': [{'id': 3, 'title': 'AI/ML Workshop', 'description': 'Workshop covering machine learning and artificial intelligence.', 'event_type': 'workshop', 'date': '2026-08-27', 'start_time': '14:00', 'end_time': '16:00', 'location': 'AI Lab', 'status': 'scheduled', 'created_at': '2026-08-26 05:53:34'}]}, 'response': 'Tomorrow, you have one event scheduled:\n\n* **AI/ML Workshop**\n  * **Time:** 2:00 PM – 4:00 PM\n  * **Location:** AI Lab\n  * **Description:** Workshop covering machine learning and artificial intelligence.\n\nThe rest of your day is currently free!'}


In [40]:
print("Your current agent link:")
print(public_url.public_url + "/chat-ui")

Your current agent link:
https://immovably-caravan-breeches.ngrok-free.dev/chat-ui


In [41]:
# TEST 1 — Check the chat UI directly

import requests

url = public_url.public_url + "/chat-ui"

response = requests.get(url)

print("URL:", url)
print("Status code:", response.status_code)
print()
print("Response:")
print(response.text[:2000])

INFO:     34.7.143.23:0 - "GET /chat-ui HTTP/1.1" 500 Internal Server Error


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 422, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        self.scope, self.receive, self.send
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/middleware/proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/fastapi/applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/applications.py", line 96, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/middleware/errors.py", line 186, i

URL: https://immovably-caravan-breeches.ngrok-free.dev/chat-ui
Status code: 500

Response:
Internal Server Error


In [42]:
# TEST 2 — Check FastAPI

response = requests.get(
    public_url.public_url + "/"
)

print("Status:", response.status_code)
print(response.text)

INFO:     34.7.143.23:0 - "GET / HTTP/1.1" 200 OK
Status: 200
{"status":"running","message":"Agentic RAG Schedule Assistant is running.","interface":"/chat-ui"}


In [43]:
# TEST 3 — Check local FastAPI server

response = requests.get(
    "http://127.0.0.1:8000/chat-ui"
)

print("Status:", response.status_code)
print()
print(response.text[:2000])

INFO:     127.0.0.1:59648 - "GET /chat-ui HTTP/1.1" 500 Internal Server Error


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 422, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        self.scope, self.receive, self.send
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/middleware/proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/fastapi/applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/applications.py", line 96, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/middleware/errors.py", line 186, i

Status: 500

Internal Server Error


In [44]:
import os

path = f"{PROJECT_DIR}/templates/index.html"

print("File exists:", os.path.exists(path))
print("Path:", path)

File exists: True
Path: /content/schedule_agent/templates/index.html


In [46]:
# CELL 33 — Check the local chat page

import requests

response = requests.get(
    "http://127.0.0.1:8000/chat-ui"
)

print("Status code:", response.status_code)
print()
print("Response:")
print(response.text[:2000])


INFO:     127.0.0.1:59638 - "GET /chat-ui HTTP/1.1" 500 Internal Server Error


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 422, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        self.scope, self.receive, self.send
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/middleware/proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/fastapi/applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/applications.py", line 96, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/middleware/errors.py", line 186, i

Status code: 500

Response:
Internal Server Error


In [47]:
# CELL 34 — Check FastAPI

response = requests.get(
    "http://127.0.0.1:8000/"
)

print("Status code:", response.status_code)
print()
print(response.text)

INFO:     127.0.0.1:33880 - "GET / HTTP/1.1" 200 OK
Status code: 200

{"status":"running","message":"Agentic RAG Schedule Assistant is running.","interface":"/chat-ui"}


In [48]:
print("YOUR AGENT PAGE:")
print(public_url.public_url + "/chat-ui")

YOUR AGENT PAGE:
https://immovably-caravan-breeches.ngrok-free.dev/chat-ui


In [49]:
# CELL 35 — Test the PUBLIC ngrok connection

import requests

public_base = public_url.public_url

print("Public URL:")
print(public_base)

print("\nTesting /health...")

health_response = requests.get(
    public_base + "/health",
    timeout=30
)

print("Health status:", health_response.status_code)
print("Health response:", health_response.text)

print("\nTesting /chat-ui...")

ui_response = requests.get(
    public_base + "/chat-ui",
    timeout=30
)

print("UI status:", ui_response.status_code)
print("UI response:")
print(ui_response.text[:1000])

Public URL:
https://immovably-caravan-breeches.ngrok-free.dev

Testing /health...
INFO:     34.7.143.23:0 - "GET /health HTTP/1.1" 200 OK
Health status: 200
Health response: {"status":"healthy"}

Testing /chat-ui...
INFO:     34.7.143.23:0 - "GET /chat-ui HTTP/1.1" 500 Internal Server Error


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 422, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        self.scope, self.receive, self.send
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/middleware/proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/fastapi/applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/applications.py", line 96, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/middleware/errors.py", line 186, i

UI status: 500
UI response:
Internal Server Error


In [50]:
# FIX — Serve index.html directly

from fastapi.responses import HTMLResponse

@app.get("/agent", response_class=HTMLResponse)
def agent_page():

    html_path = f"{PROJECT_DIR}/templates/index.html"

    with open(html_path, "r", encoding="utf-8") as f:
        html_content = f.read()

    return HTMLResponse(content=html_content)


print("New agent route created successfully!")
print()
print("LOCAL:")
print("http://127.0.0.1:8000/agent")

print()
print("PUBLIC:")
print(public_url.public_url + "/agent")

New agent route created successfully!

LOCAL:
http://127.0.0.1:8000/agent

PUBLIC:
https://immovably-caravan-breeches.ngrok-free.dev/agent


In [51]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/agent"
)

print("Status:", response.status_code)
print(response.text[:200])


INFO:     127.0.0.1:59526 - "GET /agent HTTP/1.1" 200 OK
Status: 200

<!DOCTYPE html>
<html lang="en">

<head>

<meta charset="UTF-8">

<meta name="viewport"
      content="width=device-width, initial-scale=1.0">

<title>AI Schedule Assistant</title>

<link rel="styles


In [52]:
agent_url = public_url.public_url + "/agent"

response = requests.get(
    agent_url,
    timeout=30
)

print("Agent URL:")
print(agent_url)

print()
print("Status:", response.status_code)
print(response.text[:300])

INFO:     34.7.143.23:0 - "GET /agent HTTP/1.1" 200 OK
Agent URL:
https://immovably-caravan-breeches.ngrok-free.dev/agent

Status: 200

<!DOCTYPE html>
<html lang="en">

<head>

<meta charset="UTF-8">

<meta name="viewport"
      content="width=device-width, initial-scale=1.0">

<title>AI Schedule Assistant</title>

<link rel="stylesheet"
      href="/static/style.css">

</head>

<body>

<div class="container">

    <div class="hea


In [53]:
print("OPEN THIS LINK:")
print(public_url.public_url + "/agent")

OPEN THIS LINK:
https://immovably-caravan-breeches.ngrok-free.dev/agent
